In [1]:
import pandas as pd
import numpy as np
import joblib
import threading
import requests
import time

from flask import Flask, render_template_string, request, jsonify
from werkzeug.serving import make_server

import warnings
warnings.filterwarnings('ignore')

In [2]:
gradient_boosting = joblib.load(
    "gradient_boosting_model.pkl"
)

ordinal_encoder = joblib.load(
    "ordinal_encoder.pkl"
)

preprocessor = joblib.load(
    "preprocessor.pkl"
)

categorical_features = joblib.load(
    "categorical_features.pkl"
)

numerical_features = joblib.load(
    "numerical_features.pkl"
)

binary_features = joblib.load(
    "binary_features.pkl"
)


print("Model loaded.")
print("Categorical Features:", categorical_features)
print("Numerical Features  :", numerical_features)
print("Binary Features     :", binary_features)

Model loaded.
Categorical Features: ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType']
Numerical Features  : ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']
Binary Features     : ['Weekend']


In [3]:
HTML_FORM = """
<!DOCTYPE html>
<html>

<head>

<meta charset="UTF-8">

<title>Online Purchase Intention</title>

<style>

body {
    font-family: Arial, sans-serif;
    background: #f4f6f8;
    margin: 0;
    padding: 40px 20px;
}

.container {
    max-width: 900px;
    margin: auto;
    background: white;
    padding: 30px;
    border-radius: 10px;
}

h1 {
    text-align: center;
    color: #2c3e50;
}

.subtitle {
    text-align: center;
    color: #7f8c8d;
    margin-bottom: 30px;
}

form {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 15px 20px;
}

.field {
    display: flex;
    flex-direction: column;
}

label {
    margin-bottom: 5px;
    font-weight: 600;
}

input,
select {
    padding: 9px;
    border: 1px solid #cfd8dc;
    border-radius: 6px;
}

.submit-row {
    grid-column: 1 / -1;
    margin-top: 15px;
}

button {
    width: 100%;
    padding: 12px;
    background: #2980b9;
    color: white;
    border: none;
    border-radius: 6px;
    font-size: 15px;
    cursor: pointer;
}

.result {
    margin-top: 25px;
    padding: 20px;
    text-align: center;
    border-radius: 8px;
}

.result.positive {
    background: #eafaf1;
    color: #1e8449;
}

.result.negative {
    background: #fef2f2;
    color: #c0392b;
}

.probability {
    margin-top: 8px;
    font-size: 18px;
    font-weight: bold;
}

.error {
    margin-top: 20px;
    padding: 15px;
    background: #fdecea;
    color: #c0392b;
    border-radius: 6px;
}

</style>

</head>

<body>

<div class="container">

<h1>Online Purchase Intention Predictor</h1>

<p class="subtitle">
Gradient Boosting model for predicting online purchase intention
</p>

<form action="/predict" method="POST">

    <div class="field">
        <label>Administrative Pages</label>
        <input type="number" name="Administrative" min="0" step="1" required>
    </div>

    <div class="field">
        <label>Administrative Duration</label>
        <input type="number" name="Administrative_Duration" min="0" step="any" required>
    </div>

    <div class="field">
        <label>Informational Pages</label>
        <input type="number" name="Informational" min="0" step="1" required>
    </div>

    <div class="field">
        <label>Informational Duration</label>
        <input type="number" name="Informational_Duration" min="0" step="any" required>
    </div>

    <div class="field">
        <label>Product Related Pages</label>
        <input type="number" name="ProductRelated" min="0" step="1" required>
    </div>

    <div class="field">
        <label>Product Related Duration</label>
        <input type="number" name="ProductRelated_Duration" min="0" step="any" required>
    </div>

    <div class="field">
        <label>Bounce Rate</label>
        <input type="number" name="BounceRates" min="0" max="1" step="any" required>
    </div>

    <div class="field">
        <label>Exit Rate</label>
        <input type="number" name="ExitRates" min="0" max="1" step="any" required>
    </div>

    <div class="field">
        <label>Page Value</label>
        <input type="number" name="PageValues" min="0" step="any" required>
    </div>

    <div class="field">
        <label>Special Day</label>
        <input type="number" name="SpecialDay" min="0" max="1" step="any" required>
    </div>

    <div class="field">
        <label>Month</label>
        <select name="Month" required>

            {% for value in month_values %}
            <option value="{{ value }}">{{ value }}</option>
            {% endfor %}

        </select>
    </div>

    <div class="field">
        <label>Operating System</label>
        <select name="OperatingSystems" required>

            {% for value in operating_system_values %}
            <option value="{{ value }}">{{ value }}</option>
            {% endfor %}

        </select>
    </div>

    <div class="field">
        <label>Browser</label>
        <select name="Browser" required>

            {% for value in browser_values %}
            <option value="{{ value }}">{{ value }}</option>
            {% endfor %}

        </select>
    </div>

    <div class="field">
        <label>Region</label>
        <select name="Region" required>

            {% for value in region_values %}
            <option value="{{ value }}">{{ value }}</option>
            {% endfor %}

        </select>
    </div>

    <div class="field">
        <label>Traffic Type</label>
        <select name="TrafficType" required>

            {% for value in traffic_type_values %}
            <option value="{{ value }}">{{ value }}</option>
            {% endfor %}

        </select>
    </div>

    <div class="field">
        <label>Visitor Type</label>
        <select name="VisitorType" required>

            {% for value in visitor_type_values %}
            <option value="{{ value }}">{{ value }}</option>
            {% endfor %}

        </select>
    </div>

    <div class="field">

        <label>Weekend</label>

        <select name="Weekend" required>

            <option value="0">No</option>
            <option value="1">Yes</option>

        </select>

    </div>

    <div class="submit-row">

        <button type="submit">
            Predict Purchase Intention
        </button>

    </div>

</form>


{% if result %}

<div class="result {{ 'positive' if result.prediction == 1 else 'negative' }}">

    <strong>
        {{ result.label }}
    </strong>

    <div class="probability">
        Purchase Probability: {{ result.probability }}%
    </div>

</div>

{% endif %}


{% if error %}

<div class="error">
    Error: {{ error }}
</div>

{% endif %}

</div>

</body>

</html>
"""


flask_app = Flask(__name__)

In [4]:
def prepare_input(form_data):

    input_data = pd.DataFrame({

        'Administrative': [
            int(form_data['Administrative'])
        ],

        'Administrative_Duration': [
            float(form_data['Administrative_Duration'])
        ],

        'Informational': [
            int(form_data['Informational'])
        ],

        'Informational_Duration': [
            float(form_data['Informational_Duration'])
        ],

        'ProductRelated': [
            int(form_data['ProductRelated'])
        ],

        'ProductRelated_Duration': [
            float(form_data['ProductRelated_Duration'])
        ],

        'BounceRates': [
            float(form_data['BounceRates'])
        ],

        'ExitRates': [
            float(form_data['ExitRates'])
        ],

        'PageValues': [
            float(form_data['PageValues'])
        ],

        'SpecialDay': [
            float(form_data['SpecialDay'])
        ],

        'Month': [
            form_data['Month']
        ],

        'OperatingSystems': [
            str(form_data['OperatingSystems'])
        ],

        'Browser': [
            str(form_data['Browser'])
        ],

        'Region': [
            str(form_data['Region'])
        ],

        'TrafficType': [
            str(form_data['TrafficType'])
        ],

        'VisitorType': [
            form_data['VisitorType']
        ],

        'Weekend': [
            int(form_data['Weekend'])
        ]

    })

    return input_data

In [5]:
@flask_app.route("/", methods=["GET"])
def home():

    month_values = list(
        ordinal_encoder.categories_[0]
    )

    operating_system_values = list(
        ordinal_encoder.categories_[1]
    )

    browser_values = list(
        ordinal_encoder.categories_[2]
    )

    region_values = list(
        ordinal_encoder.categories_[3]
    )

    traffic_type_values = list(
        ordinal_encoder.categories_[4]
    )

    visitor_type_values = list(
        ordinal_encoder.categories_[5]
    )

    return render_template_string(

        HTML_FORM,

        month_values=month_values,
        operating_system_values=operating_system_values,
        browser_values=browser_values,
        region_values=region_values,
        traffic_type_values=traffic_type_values,
        visitor_type_values=visitor_type_values

    )


@flask_app.route("/predict", methods=["POST"])
def predict():

    try:

        # --------------------------------------------------
        # Create input dataframe
        # --------------------------------------------------

        input_df = prepare_input(
            request.form
        )


        # --------------------------------------------------
        # Ordinal encoding
        # --------------------------------------------------

        input_encoded = input_df.copy()

        input_encoded[categorical_features] = (
            ordinal_encoder.transform(
                input_df[categorical_features]
            )
        )


        # --------------------------------------------------
        # One-hot preprocessing
        # --------------------------------------------------

        input_processed = preprocessor.transform(
            input_encoded
        )


        # --------------------------------------------------
        # Prediction
        # --------------------------------------------------

        prediction = int(
            gradient_boosting.predict(
                input_processed
            )[0]
        )

        probability = float(
            gradient_boosting.predict_proba(
                input_processed
            )[0][1]
        )


        # --------------------------------------------------
        # Result
        # --------------------------------------------------

        result = {

            "prediction": prediction,

            "label": (
                "Likely to Purchase"
                if prediction == 1
                else
                "Unlikely to Purchase"
            ),

            "probability": round(
                probability * 100,
                2
            )

        }


        return render_template_string(

            HTML_FORM,

            month_values=list(
                ordinal_encoder.categories_[0]
            ),

            operating_system_values=list(
                ordinal_encoder.categories_[1]
            ),

            browser_values=list(
                ordinal_encoder.categories_[2]
            ),

            region_values=list(
                ordinal_encoder.categories_[3]
            ),

            traffic_type_values=list(
                ordinal_encoder.categories_[4]
            ),

            visitor_type_values=list(
                ordinal_encoder.categories_[5]
            ),

            result=result

        )

    except Exception as e:

        return render_template_string(

            HTML_FORM,

            month_values=list(
                ordinal_encoder.categories_[0]
            ),

            operating_system_values=list(
                ordinal_encoder.categories_[1]
            ),

            browser_values=list(
                ordinal_encoder.categories_[2]
            ),

            region_values=list(
                ordinal_encoder.categories_[3]
            ),

            traffic_type_values=list(
                ordinal_encoder.categories_[4]
            ),

            visitor_type_values=list(
                ordinal_encoder.categories_[5]
            ),

            error=str(e)

        )

In [6]:
@flask_app.route("/api/predict", methods=["POST"])
def api_predict():

    try:

        data = request.get_json(
            force=True
        )

        input_df = prepare_input(
            data
        )


        # --------------------------------------------------
        # Ordinal encoding
        # --------------------------------------------------

        input_encoded = input_df.copy()

        input_encoded[categorical_features] = (
            ordinal_encoder.transform(
                input_df[categorical_features]
            )
        )


        # --------------------------------------------------
        # Preprocessing
        # --------------------------------------------------

        input_processed = preprocessor.transform(
            input_encoded
        )


        # --------------------------------------------------
        # Prediction
        # --------------------------------------------------

        prediction = int(
            gradient_boosting.predict(
                input_processed
            )[0]
        )

        probability = float(
            gradient_boosting.predict_proba(
                input_processed
            )[0][1]
        )


        return jsonify({

            "prediction": prediction,

            "label": (
                "Likely to Purchase"
                if prediction == 1
                else
                "Unlikely to Purchase"
            ),

            "probability": round(
                probability * 100,
                2
            )

        })

    except Exception as e:

        return jsonify({
            "error": str(e)
        }), 400

In [7]:
print("Flask app defined.")

Flask app defined.


In [8]:
# ==========================================================
# 4. START FLASK SERVER
# ==========================================================

class ServerThread(threading.Thread):

    def __init__(
        self,
        app,
        host="0.0.0.0",
        port=5000
    ):

        super().__init__()

        self.server = make_server(
            host,
            port,
            app
        )

        self.ctx = app.app_context()

        self.ctx.push()


    def run(self):

        print(
            "Flask server running... "
            "open http://localhost:5000"
        )

        self.server.serve_forever()


    def shutdown(self):

        self.server.shutdown()


server_thread = ServerThread(
    flask_app,
    port=5000
)

server_thread.start()

time.sleep(1)

Flask server running... open http://localhost:5000


In [9]:
# ==========================================================
# 5. TEST API
# ==========================================================

sample_session = {

    "Administrative": 2,
    "Administrative_Duration": 40.0,

    "Informational": 1,
    "Informational_Duration": 20.0,

    "ProductRelated": 15,
    "ProductRelated_Duration": 500.0,

    "BounceRates": 0.02,
    "ExitRates": 0.04,

    "PageValues": 10.0,

    "SpecialDay": 0.0,

    "Month": "May",

    "OperatingSystems": "2",

    "Browser": "2",

    "Region": "1",

    "TrafficType": "2",

    "VisitorType": "Returning_Visitor",

    "Weekend": 0

}


response = requests.post(
    "http://localhost:5000/api/predict",
    json=sample_session
)

print(response.status_code)
print(response.json())

127.0.0.1 - - [11/Sep/2026 19:01:59] "POST /api/predict HTTP/1.1" 200 -


200
{'label': 'Likely to Purchase', 'prediction': 1, 'probability': 57.04}


In [10]:
# ==========================================================
# 6. OPEN WEB FORM
# ==========================================================

print(
    "Open http://localhost:5000 "
    "in your browser."
)

Open http://localhost:5000 in your browser.


In [11]:
# ==========================================================
# 7. STOP SERVER
# ==========================================================

server_thread.shutdown()

print("Server stopped.")

Server stopped.
